# EDA into yfinance

In [1]:
# Cell 1 — Pull raw data from yfinance for one ticker
import yfinance as yf
import pandas as pd

raw = yf.download("VFIAX", period="1y", auto_adjust=True)
raw = raw.reset_index()
print(raw.shape)      # How many rows and columns?
raw.head(10)          # What does the data actually look like?

[*********************100%***********************]  1 of 1 completed

(250, 6)


Price,Date,Close,High,Low,Open,Volume
Ticker,,VFIAX,VFIAX,VFIAX,VFIAX,VFIAX
0,2025-03-10,513.121155,513.121155,513.121155,513.121155,0
1,2025-03-11,509.238556,509.238556,509.238556,509.238556,0
2,2025-03-12,511.738007,511.738007,511.738007,511.738007,0
3,2025-03-13,504.674194,504.674194,504.674194,504.674194,0
4,2025-03-14,515.511963,515.511963,515.511963,515.511963,0
5,2025-03-17,518.851257,518.851257,518.851257,518.851257,0
6,2025-03-18,513.338440,513.338440,513.338440,513.338440,0
7,2025-03-19,518.890747,518.890747,518.890747,518.890747,0
8,2025-03-20,517.813904,517.813904,517.813904,517.813904,0


In [2]:
# Cell 2 — Inspect the data types of each column
# This is critical for Silver — you need to know what types
# yfinance actually returns vs what you need them to be
raw.dtypes

Price   Ticker
Date              datetime64[ns]
Close   VFIAX            float64
High    VFIAX            float64
Low     VFIAX            float64
Open    VFIAX            float64
Volume  VFIAX              int64
dtype: object

In [3]:
# Cell 3 — Check for nulls across every column
raw.isnull().sum()

Price   Ticker
Date              0
Close   VFIAX     0
High    VFIAX     0
Low     VFIAX     0
Open    VFIAX     0
Volume  VFIAX     0
dtype: int64

In [4]:
# Cell 4 — Look at basic statistics to spot anomalies
# Are there any suspiciously low prices? Any zeros?
raw.describe()

Price,Date,Close,High,Low,Open,Volume
Ticker,,VFIAX,VFIAX,VFIAX,VFIAX,VFIAX
count,250,250.000000,250.000000,250.000000,250.000000,250.0
mean,2025-09-05 15:50:24,586.592740,586.592740,586.592740,586.592740,0.0
min,2025-03-10 00:00:00,455.821259,455.821259,455.821259,455.821259,0.0
25%,2025-06-06 18:00:00,547.662155,547.662155,547.662155,547.662155,0.0
50%,2025-09-06 12:00:00,597.797729,597.797729,597.797729,597.797729,0.0
75%,2025-12-03 18:00:00,630.295456,630.295456,630.295456,630.295456,0.0
max,2026-03-06 00:00:00,644.580017,644.580017,644.580017,644.580017,0.0
std,NaN,48.712037,48.712037,48.712037,48.712037,0.0


In [9]:
# Cell 5 — Now read from your actual Bronze Parquet file
# This is more important than raw yfinance because Silver
# reads from Bronze, not directly from yfinance
import duckdb

conn = duckdb.connect()
df = conn.execute("SELECT * FROM '../data/bronze/*.parquet'").df()
print(df.shape)
print(df.dtypes)
df.head(10)

(2271, 11)
Date             datetime64[ns]
Close                   float64
High                    float64
Low                     float64
Open                    float64
Volume                    int64
ticker_symbol            object
ingested_at              object
source_system            object
batch_id                 object
ingested_date            object
dtype: object


,Date,Close,High,Low,Open,Volume,ticker_symbol,ingested_at,source_system,batch_id,ingested_date
0,2025-03-10,513.121155,513.121155,513.121155,513.121155,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
1,2025-03-11,509.238556,509.238556,509.238556,509.238556,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
2,2025-03-12,511.738007,511.738007,511.738007,511.738007,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
3,2025-03-13,504.674194,504.674194,504.674194,504.674194,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
4,2025-03-14,515.512024,515.512024,515.512024,515.512024,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
5,2025-03-17,518.851257,518.851257,518.851257,518.851257,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
6,2025-03-18,513.338501,513.338501,513.338501,513.338501,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
7,2025-03-19,518.890747,518.890747,518.890747,518.890747,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
8,2025-03-20,517.813904,517.813904,517.813904,517.813904,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09
9,2025-03-21,518.258545,518.258545,518.258545,518.258545,0,VFIAX,2026-03-09T12:59:45.511922,yfinance,2026-03-09_125945,2026-03-09


In [7]:
# Cell 6 — Check what columns came through in the Parquet file
# Sometimes Parquet serialization changes things slightly
df.columns.tolist()

['Date',
 'Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'ticker_symbol',
 'ingested_at',
 'source_system',
 'batch_id',
 'ingested_date']